# 01 — Run the MVP pipeline

One cell per step of `docs/PROMPTS.md` Prompt 1: pre-flight checks (1A), start the full run in the
background (1B), tail its log, then validate the output (1C). Each cell prints a `PASS`/`FAIL` line so
the notebook can be read top-to-bottom as a checklist. No pipeline logic lives here — this only calls
`pytest`, `python -m src.run_pipeline` and `utils/validate_submission.py` via `subprocess`.

**Only one pipeline process at a time** (a full run needs most of the machine's RAM). The pre-flight
cell checks for one and refuses to start a second.


In [ ]:
import pathlib, sys

NB_DIR = pathlib.Path.cwd().resolve()
_d = NB_DIR
while not (_d / "CLAUDE.md").exists():
    if _d == _d.parent:
        raise RuntimeError("could not find student_resource/ (looked for a CLAUDE.md marker)")
    _d = _d.parent
REPO_ROOT = _d
PKG_DIR = REPO_ROOT / "code" / "business_entity_resolution"
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
DATA_DIR = REPO_ROOT / "dataset"
OUT_DIR = REPO_ROOT / "output"
print("repo root:", REPO_ROOT)


In [ ]:
import os, subprocess, sys, time


## 1A — Pre-flight checks

In [ ]:
ok = True

# 1. no other pipeline process already running
if os.name == "nt":
    out = subprocess.run(["tasklist", "/FI", "IMAGENAME eq python.exe"], capture_output=True, text=True).stdout
    running = out.lower().count("python.exe")
else:
    out = subprocess.run(["ps", "-eo", "pid,command"], capture_output=True, text=True).stdout
    running = sum(1 for line in out.splitlines() if "run_pipeline" in line)
print(("PASS" if running == 0 else "FAIL") + f": python processes detected = {running}"
      + ("" if running == 0 else " (stop before starting a new run)"))
ok &= running == 0

# 2. current commit + experiments/log.csv has rows
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True).stdout.strip()
print(f"PASS: HEAD = {commit}")
log_path = REPO_ROOT / "experiments" / "log.csv"
n_rows = max(0, sum(1 for _ in open(log_path, encoding="utf-8")) - 1) if log_path.exists() else 0
print(("PASS" if n_rows > 0 else "FAIL") + f": experiments/log.csv has {n_rows} experiment row(s)")
ok &= n_rows > 0

# 3. tests
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=PKG_DIR, capture_output=True, text=True)
print(("PASS" if r.returncode == 0 else "FAIL") + ": pytest -> " + r.stdout.strip().splitlines()[-1])
ok &= r.returncode == 0

# 4. expected caches present
expected = ["v1_translit.tsv", "v1_norm_train_source1.parquet", "v1_norm_test_source1.parquet",
            "v1_norm_test_source2.parquet", "v1_norm_test_source3.parquet", "v1_generic_vocab.json"]
cache_dir = PKG_DIR / "cache"
missing = [f for f in expected if not (cache_dir / f).exists()]
print(("PASS" if not missing else "FAIL") + f": expected caches present in {cache_dir}"
      + ("" if not missing else f" (missing: {missing})"))
ok &= not missing

print("\n" + ("ALL CHECKS PASS - safe to start the run below" if ok else "SOME CHECKS FAILED - resolve before starting a run"))


## 1B — Start the run in the background

In [ ]:
LOG_PATH = REPO_ROOT / "mvp_run.log"
env = dict(os.environ, ER_JOBS="4", ER_LGB_JOBS=str(os.cpu_count() or 4))

if not ok:
    print("FAIL: pre-flight checks did not all pass; not starting a run")
else:
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            [sys.executable, "-u", "-m", "src.run_pipeline", "--data", str(DATA_DIR), "--out", str(OUT_DIR),
             "--frac", "0.1", "--no-stage2", "--no-transfer", "--exp-id", "E03", "--change", "MVP run (notebook)"],
            cwd=PKG_DIR, stdout=f, stderr=subprocess.STDOUT, env=env,
        )
    print(f"PASS: started pid={proc.pid}, logging to {LOG_PATH}")


## Tail the log

In [ ]:
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-30:]))
    bad = [l for l in lines if any(w in l for w in ("Traceback", "Error", "WinError", "MemoryError"))]
    print("\n" + ("FAIL: error lines found:\n" + "\n".join(bad[-5:]) if bad else "PASS: no error lines so far"))
else:
    print("FAIL: log not found yet -", LOG_PATH)


## 1C — Validator + QA summary
Re-run the tail cell above until the log shows `validator exit=...`, then run this cell.

In [ ]:
r = subprocess.run([sys.executable, "utils/validate_submission.py",
                     "--matching", "output/matching_results.tsv",
                     "--candidate", "output/candidate_pairs.tsv",
                     "--test-dir", "dataset/test"],
                    cwd=REPO_ROOT, capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
print("PASS: validator exit=0" if r.returncode == 0 else f"FAIL: validator exit={r.returncode}")


In [ ]:
qa_docs = sorted(REPO_ROOT.glob("docs/QA_*.md"))
if not qa_docs:
    print("no docs/QA_*.md found (none written yet)")
else:
    for p in qa_docs:
        print(f"--- {p.name} ---")
        print(p.read_text(encoding='utf-8')[:2000])
